# Modules

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
pio.templates.default = "plotly_dark"

import sys
sys.path.append("..")
from src.preprocessing import align_densities, align_and_normalize_density
from src.utils import mark_repeated_columns
from src.plotting import px_select_menu, style_modes_of_variation, plot_acf_pacf, plot_ccfs, plot_density_timeseries_3d
from src.transformations import (
                            lqd2dens,
                            mLQDT,
                            obtain_densities_from_lqd,
                            modes_of_variation,
                            compute_lqd_cut,
                            _mode_of_variation
                            )
from src.dynamicFPC import K_dFPC, super_fun
from src.forecasting import expanding_window_cv, cv, check_autocorrelation, check_stationarity, FunctionalStationarityTest, dynamics_forecaster
from src.kde import df_bandwidth_selector, df_to_kde, weigh_norm_densities

# Session configurations
lqd_plots_xaxis = [-0.05, 1.05]

# For html export
import plotly.io as pio
pio.renderers.default = "notebook_connected"

# Params

In [ ]:
params = {
    'kernel':    't-student',
    'method':    'adaptive',
    'df': 3
    }

pp_params = {'mean_mode': 'rolling', 'window': 5}

normalize_densities = False

dimensions = 2
KdFPC_kwargs = {
    "dimension": 5, # for viz
    "lag_max": 5,
    "alpha": 0.10,
    "du": 0.05,
    "B": 1000,
    "p": 5,
    # "u": df_lqds_support,
    "select_ncomp": False
}

# Data

In [ ]:
# Data
data_path = "../data/processed/"
returns_path = ''.join([data_path, 'ibovespa_treated.xlsx'])
df = pd.read_excel(returns_path, index_col="time")
df

# In-sample analysis

## KDE

In [ ]:
df_h = df_bandwidth_selector(df, **params)
kde_params = {k: v for k, v in params.items() if k in ['kernel', 'df']}

df_support, df_densities = df_to_kde(df, df_h, **kde_params, normalize_densities=normalize_densities)
base_grid = np.linspace(df_support.min().min(), df_support.max().max(), 5001)
df_densities = weigh_norm_densities(
                                df_densities=df_densities,
                                support=base_grid,
                                **pp_params
                                )

# 2D VIZ
fig = px.line(df_densities, title=f"KDE: {params}")
px_select_menu(fig)
fig.show()

In [ ]:
# 3D VIZ
plot_density_timeseries_3d(
    df_densities,
    x_range = (-0.005, 0.005),
    colorscale="spectral"
    )

In [ ]:
df_densities.loc[:, ["2025-04-10","2025-11-28"]].plot()

In [ ]:
df.loc[:, ["2025-04-10","2025-11-28"]].to_clipboard()

In [ ]:
from scipy.integrate import trapezoid
transformed = df_densities.loc[:,"2025-04-10"]
area = trapezoid(transformed, df_densities.index)
area

## LQD

In [ ]:
bovespa_mLQDT = mLQDT(
                    df_densities,
                    df_support
                )
bovespa_mLQDT.densities_to_lqdensities(verbose=False)

bovespa_mLQDT.lqd.index = bovespa_mLQDT.lqd_support
df_lqds = bovespa_mLQDT.lqd.copy()

# 2D VIZ
fig = px.line(df_lqds, title=f"LQD from {params}")
px_select_menu(fig)
fig.update_xaxes(range=lqd_plots_xaxis)
fig.show()

In [ ]:
# 3D VIZ
# plot_density_timeseries_3d(
#     df_lqds.iloc[1:-1,:],
#     x_range = (-0.05, 1.05),
#     colorscale="spectral"
#     )

## KdFPC estimation

In [ ]:
df_lqds = bovespa_mLQDT.lqd.copy()
df_lqds_support = bovespa_mLQDT.lqd_support.copy()

KdFPC_kwargs.update({"u": df_lqds_support, "du": df_lqds_support[1] - df_lqds_support[0]})
KdFPC_model = K_dFPC(df_lqds.values)
KdFPC_model.fit(**KdFPC_kwargs)
k_scores = KdFPC_model.etahat
df_estimated = pd.DataFrame(KdFPC_model.fitted_values, index=df_lqds_support, columns=df_lqds.columns)

### Accuracy

In [ ]:
date_to_visualize = '2024-12-09'

fig = go.Figure()
fig.add_trace(go.Scatter(x=df_lqds.index, y=df_lqds[date_to_visualize], 
                         mode='lines', name='mLQDT'))
fig.add_trace(go.Scatter(x=df_estimated.index, y=df_estimated[date_to_visualize], 
                         mode='lines', name='dFPC-Estimated mLQDT'))
fig.update_layout(title="mLQDT vs dFPC estimation", xaxis_title="Time", yaxis_title="Value")
fig.show()

### Eigenfunctions

In [ ]:
eigenfunctions = pd.DataFrame(KdFPC_model.psihat.values, index=df_lqds_support, columns=[f"$ψ_{i}$" for i in range(1, KdFPC_model.psihat.shape[1]+1)])

fig = px.line(eigenfunctions, title=f"Eigenfunctions from KLE")
px_select_menu(fig)
fig.update_xaxes(range=lqd_plots_xaxis)
fig.show()

### Eigenvalues

In [ ]:
num_thetas = KdFPC_model.psihat.shape[1]
theta_labels = [f"$θ_{i}$" for i in range(1, num_thetas + 1)]
theta_values = KdFPC_model.thetahat.flatten()

fig = px.bar(
    x=theta_labels, 
    y=theta_values,
    title="Eigenvalues from KLE",
    labels={'x': 'Component', 'y': 'Eigenvalue'},
    text_auto='.4f'
)
fig.update_traces(marker_color='royalblue', opacity=0.8)
fig.update_layout(xaxis_title="Eigenvalues (θ)", yaxis_title="")
fig.show()

fig = px.bar(
    x=theta_labels, 
    y=np.log(theta_values),
    title="log(Eigenvalues) from KLE",
    labels={'x': 'Component', 'y': 'Eigenvalue'},
    text_auto='.4f'
)
fig.update_traces(marker_color='royalblue', opacity=0.8)
fig.update_layout(xaxis_title="Eigenvalues (θ)", yaxis_title="")
fig.show()

### FPC scores

In [ ]:
scores = pd.DataFrame(KdFPC_model.etahat.values, index=df_lqds.columns, columns=[f"$η_{i}$" for i in range(1, KdFPC_model.etahat.shape[1]+1)])

fig = px.line(scores, title=f"Scores from KLE")
px_select_menu(fig)
fig.update_layout(xaxis_title="Date", yaxis_title="")
fig.show()

In [ ]:
for eta in scores.columns:
    check_autocorrelation(
        scores.loc[:, eta], 
        name=eta, 
        verbose=True
        )

In [ ]:
for eta in scores.columns:
    check_stationarity(
        scores.loc[:, eta], 
        name=eta, 
        verbose=True
        )

In [ ]:
scores.reset_index().to_excel("../data/processed/dfpc_scores.xlsx", index=False)

### Correlation analysis

In [ ]:
# CORRELATION AT T
corr_matrix = scores.corr(method='pearson').mask(np.eye(len(scores.columns),dtype=bool))
pair = corr_matrix.stack().idxmax()
value = corr_matrix.loc[pair[0], pair[1]]
value_text = f"({pair[0].replace('$','')}, {pair[1].replace('$','')})"

# VIZ
fig = px.scatter_matrix(
    scores,
    dimensions=[i for i in scores.columns],
    title=f"Pairwise η_d | max: {value:.4} {value_text}",
    opacity=0.5,
    height=1000
)
fig.show()

In [ ]:
df_long = scores.melt(
    var_name="Component",
    value_name="Score"
)

fig = px.box(
    df_long,
    x="Component",
    y="Score",
    points="outliers",  # 👈 shows only outliers
    color="Component",
    template="plotly_dark"
)

fig.update_layout(
    title="FPC Score Distribution by Component",
    xaxis_title="Component",
    yaxis_title="Score",
    showlegend=False
)

fig.show()

In [ ]:
# CORRELATION AT T-1
scores_complement = scores.copy()
for col in scores_complement.columns:
    name = col.rstrip("$") + "_{-1}$"
    scores_complement[name] = scores_complement[col].shift(1)
scores_complement = scores_complement[scores_complement.columns.sort_values()]

corr_matrix = scores_complement.corr(method='pearson').mask(np.eye(len(scores_complement.columns),dtype=bool))
pair = corr_matrix.stack().idxmax()
value = corr_matrix.loc[pair[0], pair[1]]
value_text = f"{pair[0].replace('$','')}, {pair[1].replace('$','')}"

# VIZ
fig = px.scatter_matrix(
    scores_complement,
    dimensions=[i for i in scores_complement.columns],
    title=f"Pairwise η_d_(t-1) | max: {value:.4} ({value_text})",
    opacity=0.5,
    width=2000,
    height=2000
)
fig.show()

In [ ]:
# AUTOCORRELATION
df_for_dependence = scores.iloc[:,:2]
for d in df_for_dependence.columns:
    title = f"Correlation analysis"
    fig = plot_acf_pacf(
            scores.loc[:,d], 
            title=title,
            nlags=42
            )
    fig.show()

In [ ]:
# # CROSS CORRELATION
# for d1 in df_for_dependence.columns:
#     for d2 in df_for_dependence.columns:
#         if d1==d2:
#             continue
#         plot_ccfs(
#                 scores.loc[:,d1], 
#                 scores.loc[:,d2],
#                 nlags=42
#                 )

## Inverse mLQDT

In [ ]:
# LQDENSITY TO DENSITY
kle_bkw_supports, kle_bkw_densities = obtain_densities_from_lqd(
                                                            df_lqds,
                                                            df_lqds_support,
                                                            c_=bovespa_mLQDT.c,
                                                            t_=bovespa_mLQDT.t,
                                                            verbose=False
                                                            )
df_supp, df_f_kle, df_kle_fhat = align_densities(
                                df_support, 
                                df_densities, 
                                kle_bkw_supports, 
                                kle_bkw_densities, 
                                kle_bkw_densities.columns)


est_bkw_supports, est_bkw_densities = obtain_densities_from_lqd(
                                                            df_estimated,
                                                            df_lqds_support,
                                                            c_=bovespa_mLQDT.c,
                                                            t_=bovespa_mLQDT.t,
                                                            verbose=False
                                                            )
df_supp2, df_f_kle2, df_est_fhat = align_densities(
                                df_supp, 
                                df_f_kle, 
                                est_bkw_supports, 
                                est_bkw_densities, 
                                est_bkw_densities.columns)

In [ ]:
fig = go.Figure()

fig.add_trace(go.Scatter(x=df_supp[date_to_visualize], y=df_f_kle[date_to_visualize], 
                         mode='lines', name='KDE'))

fig.add_trace(go.Scatter(x=df_supp[date_to_visualize], y=df_kle_fhat[date_to_visualize], 
                         mode='lines', name='Inverse mLQDT'))

fig.add_trace(go.Scatter(x=df_supp2[date_to_visualize], y=df_est_fhat[date_to_visualize], 
                         mode='lines', name='Inverse estimated mLQDT'))

fig.update_layout(title="KDE vs Inverse mLQDT", xaxis_title="Time", yaxis_title="Value",hovermode="x unified",)

fig.show()

## Modes of variation / Modos de variação

\begin{equation}
    q_k(u, \alpha, \Lambda_q) = \Lambda_q^{-1}(\mu + \alpha\sqrt{\lambda_k}\psi_k)(u)
\end{equation}


* $g_k$: The $\textbf{perturbed density function}$ for the $k$-th component, representing the reconstructed density after applying a specific variation.
* $\alpha$: The $\textbf{perturbation scalar}$.
* $\mu$: The $\textbf{mean function}$ in the transformed space.
* $\lambda_k$:The $\textbf{k-th eigenvalue}$, quantifying the total variance explained by the $k$-th principal component.
* $\psi_k$: The $\textbf{k-th eigenfunction}$ (or basis function), defining the specific geometric direction or pattern of the variation.

### In L2

In [ ]:
perturbations = [0.00, 0.5, 2, 10]
modes = modes_of_variation(
    df_lqds,
    KdFPC_model.psihat.values,
    KdFPC_model.thetahat,
    alphas=perturbations
)

In [ ]:
modes.head()

In [ ]:
# VIZ MODES OF VARIATION
# fig = px.line(
#     modes,      
#     x="index",                
#     y="value",               
#     color="alpha",          
#     facet_col="pc",          
#     facet_col_wrap=2,        
#     title="Modes of Variation: Mean vs Principal Components",
#     labels={"index": "u", "value": "lqd", "alpha": "α"},
#     height=1000
# )
# fig.update_layout(
#     # hovermode="x unified",    
#     legend_title_text='Alpha Intensity',
# )
# # fig.update_xaxes(range=[-0.05, 1.05])
# style_modes_of_variation(fig, perturbations, mu_black=False, mu_transparency=0.8)
# px_select_menu(fig)
# fig.show()

### In pdf space

In [ ]:
# USE AVERAGE C TO GET THE INVERSE OF ALL MODES OF VARIATION
common_c = np.mean(bovespa_mLQDT.c)
common_t0 = bovespa_mLQDT.t[0]
common_dSup = df_support.iloc[:,0]

# CREATES DATAFRAME IN LONG FORMAT
df_dens_modes = []
for pc in modes.pc.unique():
    for alpha in modes.alpha.unique():
        mode_of_variation = modes[(modes.pc==pc)&(modes.alpha==alpha)]
        lqd = mode_of_variation.loc[:,"value"]
        cut = compute_lqd_cut(lqd, cut_nan=True)
        dSup_temp, dens_temp = lqd2dens(
                                lqd=lqd.values, 
                                lqdSup=mode_of_variation["index"], 
                                c=common_c, 
                                dSup=df_support.iloc[:,0].values, 
                                cut=cut,
                                verbose=False
                                )
        df_temp = pd.DataFrame(
            {
            "index": dSup_temp,
            "pc": pc,
            "alpha": alpha,
            "value": dens_temp
            }
        )
        df_dens_modes.append(df_temp)

df_pdf_modes = pd.concat(df_dens_modes)

In [ ]:
fig = px.line(
    df_pdf_modes,      
    x="index",                
    y="value",                
    color="alpha",            
    facet_col="pc",           
    facet_col_wrap=2,      
    title=r"Modes of Variation: Mean vs Principal Components in pdf space",
    labels={"index": "u", "value": "density", "alpha": "α"},
    height=1000
)
fig.update_layout(
    # hovermode="x unified",     
    legend_title_text='Alpha Intensity',
)
style_modes_of_variation(fig, perturbations, mu_black=False, mu_transparency=0.8)
px_select_menu(fig)
# fig.update_xaxes(range=[-0.01, 0.01])
fig.show()

In [ ]:
# Viz: largest absolute FPC score + arbitrary perturbations (TODO)
largest_score = np.max(np.abs(KdFPC_model.etahat))

_theta = (largest_score/10)**2
_alpha = 10
lower_curve, upper_curve = _mode_of_variation(
    mu=df_lqds.mean(axis=1).values,
    psi=KdFPC_model.psihat.iloc[:,0],
    theta = _theta,
    alpha=_alpha
)

import matplotlib.pyplot as plt
plt.figure(figsize=(15,5))

plt.plot(df_lqds.mean(axis=1).index, upper_curve)

plt.plot(df_lqds.mean(axis=1).index, df_lqds.mean(axis=1).values)

plt.plot(df_lqds.mean(axis=1).index, lower_curve)


plt.show()

## Horta & Ziegelmann, 2018

In [ ]:
sp = super_fun(
    Y=df_densities.values,
    lag_max=KdFPC_kwargs["lag_max"],
    B=KdFPC_kwargs["B"],
    p=KdFPC_kwargs["p"],
    m=df_densities.shape[0],
    du=0.05,
    dimension=5,
    alpha=0.05,
    u=df_support.iloc[:,0]
)
Yhat_superfun = pd.DataFrame(sp["Yhat"], columns=df_estimated.columns, index=df_support.iloc[:,0])
df_eigenfunc_superfun = pd.DataFrame(sp['psihat'], columns=eigenfunctions.columns)
df_scores_superfun    = pd.DataFrame(sp["etahat"].T, columns=scores.columns) 

In [ ]:
df_comparison_scores = scores.iloc[:,:1].reset_index().copy()

df_comparison_scores.rename(columns={"index": "date", "$η_1$": "eta_LQdFPC"}, inplace=True)

df_comparison_scores["eta_dFPC"] = df_scores_superfun.iloc[:,0].values

df_comparison_scores.to_excel("../data/processed/scores_LQdFPC_vs_dFPC.xlsx", index=False)

# Forecasting

In [ ]:
from src.forecasting import dynamics_forecaster

## Fitted model

### LQDensities

In [ ]:
tester = FunctionalStationarityTest(df_lqds, grid_vals=df_lqds_support)
tester.run_test(mc_rep=2000,d=dimensions)
tester.get_summary()

### VAR on FPC scores

In [ ]:
lags = 3

In [ ]:
ts_scores = scores.iloc[:, 0:dimensions]
ts_scores.head()

In [ ]:
# Summary of fitted model
forecaster = dynamics_forecaster(ts_scores)
forecaster.fit_var(nlags=lags)
forecaster.fitted_model.summary()

In [ ]:
coeffs =forecaster.fitted_model.coefs

n_lags = coeffs.shape[0]   # 3
n_scores = coeffs.shape[1] # 5

reshaped_values = coeffs.transpose(1, 0, 2).reshape(n_scores, n_lags * n_scores)

col_indices = pd.MultiIndex.from_product(
    [range(1, n_lags + 1), range(1, n_scores + 1)],
    names=['Lag (k)', 'Score (j)']
)

row_indices = [f'Eq for Score {i}' for i in range(1, n_scores + 1)]

df_final = pd.DataFrame(reshaped_values, index=row_indices, columns=col_indices)

df_final = df_final.round(4)

df_final

In [ ]:
# Residuals
var_resid = forecaster.fitted_model.resid
var_resid.head()

In [ ]:
# Residuals
forecaster.residual_diagnostics(plot=True)

In [ ]:
df_long = forecaster.fitted_model.resid.melt(
    var_name="Score",
    value_name="Value"
)

fig = px.box(
    df_long,
    x="Score",
    y="Value",
    points="outliers",  # 👈 shows only outliers
    color="Score",
    template="plotly_dark"
)

fig.update_layout(
    title="VAR(q) residuals",
    xaxis_title="Score",
    yaxis_title="Value",
    showlegend=False
)

fig.show()

## Expanding window

### Expanding window forecasts

#### Parameters

In [ ]:
horizon = 1
initial_window = len(df_densities.columns) - 100
KdFPC_kwargs.update({"dimension": dimensions})

#### Cross-Validation

In [ ]:
cv_results = cv(
                        df_densities, 
                        df_support, 
                        KdFPC_kwargs   = KdFPC_kwargs, 
                        horizon        = horizon, 
                        initial_window = initial_window,
                        return_curves  = True,
                        var_lags       = dimensions
                        )

df_cv_results = pd.DataFrame(cv_results)

#### Results

In [ ]:
# STORES CURVES VALUES
residuals = []
kde       = []
forecasts = []
for row in df_cv_results.iterrows():
    supp0 = row[1]["df_supports"].iloc[:,0].values
    fc0 = row[1]["df_forecast"].iloc[:,0]
    fc0.index = supp0
    kde0 = row[1]["df_kde"].iloc[:,0]
    kde0.index = supp0
    kde0.name = fc0.name
    resid0 =  kde0 - fc0
    resid0.index = supp0
    fc0.index = supp0
    residuals.append(resid0)
    kde.append(kde0)
    forecasts.append(fc0)

In [ ]:
fig = go.Figure()

for col in forecasts:
    fig.add_trace(go.Scatter(x=col.index,y=col, mode='lines', name=str(col.name)))

px_select_menu(fig)

fig.show()

In [ ]:
# # VIZ FORECAST VALUES
# df_forecasts = pd.concat(forecasts, axis=1)
# # df_forecasts.index = df_densities.index

# fig = px.line(
#     df_forecasts,
#     color_discrete_sequence=['#80ed99'],
#     title= "Forecasts from CV"
#     )
# fig.update_traces(opacity=.4)
# px_select_menu(fig)
# fig.show()

In [ ]:
fig = go.Figure()

for col in residuals:
    fig.add_trace(
        go.Scatter(
            x=col.index,
            y=col, 
            mode='lines', 
            name=str(col.name), 
            line=dict(color='royalblue', width=1),
            opacity=0.3)
            )
    
# 2. Calculate the mean line
# We stack the list into a 2D array and average across rows
mean_y = np.mean([s.values for s in residuals], axis=0)
# Use the index from the first series in the list
common_x = residuals[0].index

# 3. Add the Mean Line Trace
fig.add_trace(
    go.Scatter(
        x=common_x,
        y=mean_y,
        mode='lines',
        name='Ensemble Mean',
        line=dict(color='red', width=1, dash='dot')
    )
)
        

px_select_menu(fig)

fig.show()

In [ ]:
(pd.concat(residuals, axis=1)/pd.concat(kde, axis=1)).plot(legend=False, figsize=(15,5))

In [ ]:
# Forecast

fig = go.Figure()

for col in kde:
    fig.add_trace(
        go.Scatter(
            x=col.index,
            y=col, 
            mode='lines', 
            name=str(col.name), 
            line=dict(color='royalblue', width=1),
            opacity=0.3)
            )
    
# 2. Calculate the mean line
# We stack the list into a 2D array and average across rows
mean_y = np.mean([s.values for s in kde], axis=0)
# Use the index from the first series in the list
common_x = residuals[0].index

# 3. Add the Mean Line Trace
fig.add_trace(
    go.Scatter(
        x=common_x,
        y=mean_y,
        mode='lines',
        name='Ensemble Mean',
        line=dict(color='red', width=1, dash='dot')
    )
)
        

px_select_menu(fig)

fig.show()

In [ ]:
# VIZ RESIDUALS
# df_cv_residuals = pd.concat(residuals, axis=1)

# fig = px.line(
#     df_cv_residuals,
#     color_discrete_sequence=['#80ed99'],
#     title= "Residuals from CV forecast"
#     )
# fig.update_traces(opacity=.4)
# fig.add_scatter(
#     x=df_cv_residuals.index, 
#     y=df_cv_residuals.mean(axis=1), 
#     mode='lines',
#     name='Pointwise mean',
#     line=dict(color='orange', width=6)
# )
# px_select_menu(fig)
# fig.show()

In [ ]:
# OBTAINS BEST AND WORST FORECASTS
measures = ['KLD', 'JSD', 'L_1', 'L_2', 'L_INFTY']
best_and_worst = []
for measure in measures:
    best_forecast_id = df_cv_results[measure].idxmin()
    df_best = df_cv_results.iloc[best_forecast_id, :]
    best_info = {
        "metric":   measure,
        "position": "best",
        "fold":     df_best["fold"],
        "day":      df_best.loc["df_kde"].columns[0],
        "support":  df_best.loc["df_supports"].iloc[:,0].values,
        "kde":      df_best.loc["df_kde"].iloc[:,0].values,
        "forecast": df_best.loc["df_forecast"].iloc[:,0].values
    }
    
    best_and_worst.append(pd.DataFrame(best_info))

    worst_forecast_id = df_cv_results[measure].idxmax()
    df_worst = df_cv_results.iloc[worst_forecast_id, :]
    worst_info = {
        "metric":   measure,
        "position": "worst",
        "fold":     df_worst["fold"],
        "day":      df_worst.loc["df_kde"].columns[0],
        "support":  df_worst.loc["df_supports"].iloc[:,0].values,
        "kde":      df_worst.loc["df_kde"].iloc[:,0].values,
        "forecast": df_worst.loc["df_forecast"].iloc[:,0].values
    }
    best_and_worst.append(pd.DataFrame(worst_info))

df_best_and_worst = pd.concat(best_and_worst)
df_results_temp = df_best_and_worst[["metric", "position", "day", "support", "kde", "forecast"]]
df_results = df_best_and_worst.melt(id_vars=["metric", "position", "day","support"],value_vars=["kde", "forecast"])

# DATES FOR BEST AND WORST FORECASTS
df_best_and_worst[["metric", "position", "day"]].drop_duplicates().sort_values(by="position")

In [ ]:
# VIZ BEST AND WORST FORECASTS
fig = px.line(
    df_results,      
    x="support",                
    y="value",                
    color="variable",           
    facet_col="position",           
    facet_col_wrap=2,         
    facet_row="metric",
    title="Best and worst forecasts by each metric",
    labels={"index": "x", "value": "density", "alpha": "α"},
    height=1000,
    color_discrete_sequence=['#e63946', '#02c39a']
)
fig.update_layout(
    hovermode="x unified"
)
fig.show()